In [13]:

# ============================================================
# EGATE AI - XGBOOST TRAINING
# ============================================================
#
# m.csv
#   ↓
# preprocess data
#   ↓
# train XGBoost
#   ↓
# evaluate model
#   ↓
# save addiction_pipeline.pkl
#
# ============================================================


# ============================================================
# 1. IMPORTS
# ============================================================

import os
import glob
import joblib
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

from xgboost import XGBClassifier

















In [14]:
# ============================================================
# 2. FIND m.csv
# ============================================================

print("=" * 60)
print("SEARCHING FOR m.csv")
print("=" * 60)

possible_paths = [
    "/Users/mo/Egate_Ai_camp/m.csv",
    "/Users/mo/Egate_Ai_camp/dbpro/m.csv",
    "/Users/mo/Egate_Ai_camp/Data for big pro/m.csv",
    os.path.join(os.getcwd(), "m.csv"),
    "/mnt/data/m.csv",
]

DATA_PATH = None

for path in possible_paths:

    if os.path.isfile(path):

        DATA_PATH = path
        break


# Search entire Egate project if not found
if DATA_PATH is None:

    matches = glob.glob(
        "/Users/mo/Egate_Ai_camp/**/m.csv",
        recursive=True
    )

    if matches:

        DATA_PATH = matches[0]


if DATA_PATH is None:

    raise FileNotFoundError(
        "❌ Could not find m.csv.\n"
        "Put m.csv somewhere inside "
        "/Users/mo/Egate_Ai_camp/"
    )


print("✅ CSV found:")
print(DATA_PATH)


SEARCHING FOR m.csv
✅ CSV found:
/Users/mo/Egate_Ai_camp/dbpro/m.csv


In [15]:
# ============================================================
# 3. LOAD CSV
# ============================================================

print()
print("=" * 60)
print("LOADING DATASET")
print("=" * 60)

df = pd.read_csv(DATA_PATH)

print(
    f"Rows: {df.shape[0]}"
)

print(
    f"Columns: {df.shape[1]}"
)

print()
print(
    "Columns:"
)

for column in df.columns:

    print(
        " -",
        column
    )


LOADING DATASET
Rows: 7500
Columns: 17

Columns:
 - transaction_id
 - user_id
 - age
 - gender
 - daily_screen_time_hours
 - social_media_hours
 - gaming_hours
 - work_study_hours
 - sleep_hours
 - notifications_per_day
 - app_opens_per_day
 - weekend_screen_time
 - stress_level
 - academic_work_impact
 - addiction_level
 - addicted_label
 - nsfw_score


In [16]:

# ============================================================
# 4. TARGET
# ============================================================

TARGET = "addicted_label"

if TARGET not in df.columns:

    raise ValueError(
        f"❌ '{TARGET}' was not found in the CSV."
    )


# Convert target to integer
y = df[TARGET].astype(int)


In [17]:
# ============================================================
# 5. REMOVE BAD / LEAKING COLUMNS
# ============================================================
#
# transaction_id
#   -> identifier
#
# user_id
#   -> identifier
#
# addiction_level
#   -> directly related to the target
#      so this would leak the answer
#
# nsfw_score
#   -> your dataset currently has no useful variation
#
# addicted_label
#   -> target, so it must not be inside X
# ============================================================

DROP_COLUMNS = [
    "transaction_id",
    "user_id",
    "addiction_level",
    "nsfw_score",
    "addicted_label"
]


X = df.drop(
    columns=DROP_COLUMNS,
    errors="ignore"
)



In [18]:
 #============================================================
# 6. DISPLAY FEATURES
# ============================================================


print()
print("=" * 60)
print("FEATURES USED FOR TRAINING")
print("=" * 60)

for column in X.columns:

    print(
        " -",
        column
    )




FEATURES USED FOR TRAINING
 - age
 - gender
 - daily_screen_time_hours
 - social_media_hours
 - gaming_hours
 - work_study_hours
 - sleep_hours
 - notifications_per_day
 - app_opens_per_day
 - weekend_screen_time
 - stress_level
 - academic_work_impact


In [19]:
#7. FIND CATEGORICAL / NUMERICAL COLUMNS
# ============================================================

categorical_cols = (
    X.select_dtypes(
        include=["object", "category"]
    )
    .columns
    .tolist()
)

numerical_cols = (
    X.select_dtypes(
        include=[
            "int64",
            "int32",
            "float64",
            "float32"
        ]
    )
    .columns
    .tolist()
)


print()
print("Categorical columns:")

for column in categorical_cols:

    print(
        " -",
        column
    )


print()
print("Numerical columns:")

for column in numerical_cols:

    print(
        " -",
        column
    )









Categorical columns:
 - gender
 - stress_level
 - academic_work_impact

Numerical columns:
 - age
 - daily_screen_time_hours
 - social_media_hours
 - gaming_hours
 - work_study_hours
 - sleep_hours
 - notifications_per_day
 - app_opens_per_day
 - weekend_screen_time


/var/folders/cw/z581ry497cn3g465x_ywjhrw0000gn/T/ipykernel_43965/2552394510.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  X.select_dtypes(


In [20]:
# ============================================================
# 8. CHECK MISSING VALUES
# ============================================================

print()
print("=" * 60)
print("CHECKING MISSING VALUES")
print("=" * 60)

missing = X.isnull().sum()

missing = missing[
    missing > 0
]

if missing.empty:

    print(
        "✅ No missing feature values."
    )

else:

    print(
        missing
    )




CHECKING MISSING VALUES
✅ No missing feature values.


In [21]:
# ============================================================
# 9. PREPROCESSOR
# ============================================================

preprocessor = ColumnTransformer(

    transformers=[

        (
            "categorical",

            # One-hot encoding: each category becomes its own
            # 0/1 column (e.g. gender_Male, gender_Female) instead
            # of being squeezed into a single number. That avoids
            # implying a fake order between categories that have
            # no real order (OrdinalEncoder would have made
            # "Female" mathematically greater than "Male").
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),

            categorical_cols
        ),

        (
            "numerical",

            StandardScaler(),

            numerical_cols
        )
    ],

    remainder="drop"
)

In [22]:
# ============================================================
# 10. TRAIN / TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = (
    train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42,
        stratify=y
    )
)


print()
print("=" * 60)
print("DATA SPLIT")
print("=" * 60)

print(
    "Training rows:",
    len(X_train)
)

print(
    "Testing rows:",
    len(X_test)
)


DATA SPLIT
Training rows: 6000
Testing rows: 1500


In [23]:
# ============================================================
# 11. FIT PREPROCESSOR
# ============================================================

print()
print(
    "Preprocessing training data..."
)

X_train_encoded = (
    preprocessor.fit_transform(
        X_train
    )
)

X_test_encoded = (
    preprocessor.transform(
        X_test
    )
)

print(
    "✅ Preprocessing complete."
)





Preprocessing training data...
✅ Preprocessing complete.


In [24]:
# ============================================================
# 12. CREATE XGBOOST MODEL
# ============================================================

print()
print("=" * 60)
print("CREATING XGBOOST MODEL")
print("=" * 60)

model = XGBClassifier(

    n_estimators=200,

    max_depth=4,

    learning_rate=0.05,

    subsample=0.9,

    colsample_bytree=0.9,

    objective="binary:logistic",

    eval_metric="logloss",

    random_state=42
)



CREATING XGBOOST MODEL


In [25]:
# ============================================================
# 13. TRAIN
# ============================================================

print()
print(
    "Training XGBoost..."
)

model.fit(
    X_train_encoded,
    y_train
)

print(
    "✅ Training finished."
)











Training XGBoost...
✅ Training finished.


In [26]:
# ============================================================
# 14. PREDICTIONS
# ============================================================

y_pred = model.predict(
    X_test_encoded
)

y_probability = model.predict_proba(
    X_test_encoded
)[:, 1]


In [34]:
# ============================================================
# 15. EVALUATION
# ============================================================

accuracy = accuracy_score(
    y_test,
    y_pred
)

roc_auc = roc_auc_score(
    y_test,
    y_probability
)


print()
print("=" * 60)
print("MODEL RESULTS")
print("=" * 60)

print(
    f"Accuracy: {accuracy:.4f}"
)

print(
    f"ROC-AUC:  {roc_auc:.4f}"
)

print()
print(
    "Classification report:"
)

print(
    classification_report(
        y_test,
        y_pred
    )
)

print()
print(
    "Confusion matrix:"
)

print(
    confusion_matrix(
        y_test,
        y_pred
    )
)


MODEL RESULTS
Accuracy: 0.9387
ROC-AUC:  0.9894

Classification report:
              precision    recall  f1-score   support

           0       0.87      0.92      0.90       438
           1       0.97      0.95      0.96      1062

    accuracy                           0.94      1500
   macro avg       0.92      0.93      0.93      1500
weighted avg       0.94      0.94      0.94      1500


Confusion matrix:
[[ 404   34]
 [  58 1004]]


In [28]:

# ============================================================
# 16. FEATURE IMPORTANCE
# ============================================================

print()
print("=" * 60)
print("FEATURE IMPORTANCE")
print("=" * 60)

feature_names = (
    preprocessor
    .get_feature_names_out()
)

importance_df = pd.DataFrame({

    "feature":
        feature_names,

    "importance":
        model.feature_importances_

})

importance_df = (
    importance_df
    .sort_values(
        "importance",
        ascending=False
    )
)


print(
    importance_df.to_string(
        index=False
    )
)



FEATURE IMPORTANCE
                              feature  importance
        numerical__social_media_hours    0.359671
   numerical__daily_screen_time_hours    0.323129
       numerical__weekend_screen_time    0.146813
 categorical__academic_work_impact_No    0.016585
     categorical__stress_level_Medium    0.013956
        categorical__stress_level_Low    0.013705
            categorical__gender_Other    0.013295
          numerical__work_study_hours    0.013280
               numerical__sleep_hours    0.012744
              numerical__gaming_hours    0.012497
                       numerical__age    0.011916
         numerical__app_opens_per_day    0.011627
           categorical__gender_Female    0.011375
     numerical__notifications_per_day    0.011294
categorical__academic_work_impact_Yes    0.010951
             categorical__gender_Male    0.010623
       categorical__stress_level_High    0.006537


In [29]:
# ============================================================
# 17. SAVE MODEL
# ============================================================

MODEL_PATH = (
    "/Users/mo/Egate_Ai_camp/"
    "addiction_pipeline.pkl"
)


artifact = {

    # The preprocessing object
    "preprocessor":
        preprocessor,

    # XGBoost model
    "model":
        model,

    # Exact input columns
    "feature_columns":
        X.columns.tolist(),

    # Target name
    "target_column":
        TARGET,

    # Evaluation
    "accuracy":
        accuracy,

    "roc_auc":
        roc_auc,

    # Feature importance
    "feature_importance":
        importance_df
}


joblib.dump(
    artifact,
    MODEL_PATH
)




['/Users/mo/Egate_Ai_camp/addiction_pipeline.pkl']

In [30]:
# ============================================================
# 18. VERIFY MODEL FILE
# ============================================================

print()
print("=" * 60)
print("VERIFYING SAVED MODEL")
print("=" * 60)

if os.path.exists(
    MODEL_PATH
):

    print(
        "✅ Model successfully saved!"
    )

    print()
    print(
        "Location:"
    )

    print(
        MODEL_PATH
    )

else:

    raise RuntimeError(
        "❌ Model file was not created."
    )



VERIFYING SAVED MODEL
✅ Model successfully saved!

Location:
/Users/mo/Egate_Ai_camp/addiction_pipeline.pkl


In [31]:
# ============================================================
# 19. LOAD IT AGAIN TO MAKE SURE IT WORKS
# ============================================================

print()
print(
    "Testing saved model..."
)

loaded_artifact = joblib.load(
    MODEL_PATH
)

loaded_preprocessor = (
    loaded_artifact["preprocessor"]
)

loaded_model = (
    loaded_artifact["model"]
)


# Test with one row
sample = X_test.iloc[
    [0]
]


sample_encoded = (
    loaded_preprocessor.transform(
        sample
    )
)


sample_probability = float(
    loaded_model
    .predict_proba(
        sample_encoded
    )[0][1]
)


sample_prediction = int(
    sample_probability >= 0.50
)


print()
print(
    "Sample prediction:",
    sample_prediction
)

print(
    f"Sample risk probability: "
    f"{sample_probability:.2%}"
)





Testing saved model...

Sample prediction: 0
Sample risk probability: 0.53%


In [32]:
# ============================================================
# 20. FINAL
# ============================================================

print()
print("=" * 60)
print("✅ EGATE AI XGBOOST TRAINING COMPLETE")
print("=" * 60)

print()
print(
    "Your trained model is:"
)

print(
    MODEL_PATH
)

print()
print(
    "This .pkl file can now be loaded "
    "by background_monitor.py."
)






✅ EGATE AI XGBOOST TRAINING COMPLETE

Your trained model is:
/Users/mo/Egate_Ai_camp/addiction_pipeline.pkl

This .pkl file can now be loaded by background_monitor.py.


In [33]:
import os
print(os.getcwd())
joblib.dump(artifact, "/Users/mo/Egate_Ai_camp/dbpro/addiction_pipeline.pkl")

/Users/mo/Egate_Ai_camp/dbpro


['/Users/mo/Egate_Ai_camp/dbpro/addiction_pipeline.pkl']